# 01 — Veri Hazırlama

**İlan maddesi:** *"disiplinler arası... belge yönetimi bileşenleri"*, RAG için veri altyapısı.

Bu notebook üç adımı kapsar:
1. Türkçe Wikipedia'dan savunma/havacılık temalı, kamuya açık bir korpus toplama
2. Korpusu RAG için token-bazlı chunk'lara bölme
3. Chunk'lardan instruction-tuning (SFT) veri seti üretme (self-instruct tekniği)

Kod, `src/data_prep/` altında modüler ve test edilebilir şekilde yazıldı; burada sadece
uçtan uca çalıştırıyoruz.

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) hiçbir şey yapmadan devam eder.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
import os, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"

if not os.path.exists(PROJECT_DIR):
    try:
        from google.colab import drive
        # drive.mount() zaten mount edilmişse anında geri döner (idempotent);
        # os.path.exists("/content/drive") ile "mount edilmiş mi" kontrol etmek
        # güvenilmez çünkü klasör, başarısız/yarım bir mount denemesinden sonra
        # bile var olabilir. Bu yüzden koşulsuz çağırıyoruz.
        drive.mount("/content/drive", force_remount=True)
        if os.path.exists(DRIVE_ZIP_PATH):
            import shutil
            shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
        else:
            print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
                  "yükleyin ya da kendi reponuzu klonlayın: "
                  f"!git clone <repo-url> {PROJECT_DIR}")
    except ImportError:
        pass  # Colab dışında (yerelde) çalışıyorsanız bu adım gerekmez.

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)


## 1. Korpus toplama

Kamuya açık Wikipedia makalelerini indirip temizliyoruz.

In [ ]:
from src.data_prep.corpus_builder import build_corpus, save_corpus

corpus = build_corpus()
path = save_corpus(corpus)
print(f"{len(corpus)} doküman kaydedildi -> {path}")
for doc in corpus[:3]:
    print("-", doc.title, f"({len(doc.text)} karakter)")


## 2. Chunking

Dokümanları embedding modelinin tokenizer'ıyla uyumlu, örtüşmeli parçalara bölüyoruz.

In [ ]:
from src.data_prep.chunking import chunk_corpus, save_chunks

chunks = chunk_corpus()
chunk_path = save_chunks(chunks)
print(f"{len(chunks)} chunk kaydedildi -> {chunk_path}")
print("\nÖrnek chunk:\n", chunks[0].text[:300])


## 3. Instruction (SFT) veri seti üretimi

`use_llm=True` ile temel LLM'i kullanarak her chunk'tan bağlama sadık bir soru-cevap
çifti üretiyoruz (self-instruct). GPU yoksa ya da hızlı denemek isterseniz `use_llm=False`
şablon moduna düşer.

In [ ]:
from src.data_prep.instruction_dataset import build_sft_dataset, save_sft_dataset

sft_examples = build_sft_dataset(use_llm=True, max_examples=150)
sft_path = save_sft_dataset(sft_examples)
print(f"{len(sft_examples)} SFT örneği kaydedildi -> {sft_path}")
for ex in sft_examples[:2]:
    print("\nSoru:", ex.instruction)
    print("Cevap:", ex.output)
